## 1.Monitoring Views (Operational Layer)
These views provide a 7-day rolling operational layer on top of:
* pipeline_run_log (execution observability)
* dq_rule_result / dq_table_gate (data quality evidence)

### 1) Success Rate
(SUCCESS / DEGRADED / BLOCKED / SKIPPED / FAILED + success rate)

In [13]:
%%sql
-- # v_monitor_success_rate_7d
CREATE OR REPLACE VIEW v_monitor_success_rate_7d AS
SELECT
  pipeline_name,
  date(start_time) AS run_date,

  COUNT(*) AS total_runs,
  SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) AS success_runs,
  SUM(CASE WHEN status = 'SKIPPED' THEN 1 ELSE 0 END) AS skipped_runs,
  SUM(CASE WHEN status = 'BLOCKED' THEN 1 ELSE 0 END) AS blocked_runs,
  SUM(CASE WHEN status = 'DEGRADED' THEN 1 ELSE 0 END) AS degraded_runs,
  SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) AS failed_runs,

  ROUND(
    100.0 * SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END)
    / NULLIF(SUM(CASE WHEN status <> 'SKIPPED' THEN 1 ELSE 0 END), 0),
    2
  ) AS success_rate_excl_skipped_pct,

  ROUND(
    100.0 * SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END)
    / count(*),
    2
  ) AS success_rate_incl_skipped_pct,
  CASE
    WHEN SUM(CASE WHEN status <> 'SKIPPED' THEN 1 ELSE 0 END) = 0
      THEN 'ALL_SKIPPED'
    WHEN SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) > 0
      THEN 'FAILED'
    WHEN SUM(CASE WHEN status = 'BLOCKED' THEN 1 ELSE 0 END) > 0
      THEN 'BLOCKED'
    WHEN SUM(CASE WHEN status = 'DEGRADED' THEN 1 ELSE 0 END) > 0
      THEN 'DEGRADED'
    ELSE 'HEALTHY'
  END AS run_health_label
FROM pipeline_run_log
WHERE start_time >= date_sub(current_timestamp(), 7)
GROUP BY pipeline_name, date(start_time)
ORDER BY run_date DESC, pipeline_name;

StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### 2) Status Breakdown (SKIPPED / BLOCKED / DEGRADED / FAILED)

In [14]:
%%sql
CREATE OR REPLACE VIEW v_monitor_blocked_degraded_7d AS
SELECT
  pipeline_name,
  date(start_time) AS run_date,
  SUM(CASE WHEN status = 'SKIPPED' THEN 1 ELSE 0 END) AS skipped_cnt,
  SUM(CASE WHEN status = 'BLOCKED' THEN 1 ELSE 0 END) AS blocked_cnt,
  SUM(CASE WHEN status = 'DEGRADED' THEN 1 ELSE 0 END) AS degraded_cnt,
  SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) AS failed_cnt,
  COUNT(*) AS total_runs
FROM pipeline_run_log
WHERE start_time >= date_sub(current_timestamp(), 7)
GROUP BY pipeline_name, date(start_time)
ORDER BY run_date DESC, pipeline_name;


StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 16, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### 3) Data Quality Drilldown
(Top failing rules + drilldown by pipeline_run_id)

In [15]:
%%sql
CREATE OR REPLACE VIEW v_monitor_top_failed_rules_7d AS
SELECT
  d.layer,
  d.table_name,
  d.rule_id,
  d.rule_name,
  d.severity,
  COUNT(*) AS fail_events,
  ROUND(AVG(d.failed_rate), 4) AS avg_failed_rate,
  ROUND(AVG(d.threshold_rate), 4) AS avg_threshold_rate,
  MAX(to_timestamp(d.run_ts)) AS last_fail_ts
FROM dq_rule_result d
WHERE to_timestamp(d.run_ts) >= date_sub(current_timestamp(), 7)
  AND d.status = 'FAIL'
  AND d.severity IN ('CRITICAL', 'MAJOR')
GROUP BY d.layer, d.table_name, d.rule_id, d.rule_name, d.severity
ORDER BY fail_events DESC, last_fail_ts DESC;


StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [17]:
%%sql
CREATE OR REPLACE VIEW v_monitor_run_dq_fail_drilldown_7d AS
SELECT
  r.pipeline_run_id,
  r.pipeline_name,
  r.layer AS pipeline_layer,
  r.table_name,
  r.status AS pipeline_status,
  r.start_time,
  r.end_time,
  UNIX_TIMESTAMP(r.end_time) - UNIX_TIMESTAMP(r.start_time) AS duration_sec,
  d.table_name AS dq_table,
  d.rule_id,
  d.rule_name,
  d.severity,
  d.failed_count,
  d.total_count,
  d.failed_rate,
  d.threshold_rate,
  to_timestamp(d.run_ts) AS dq_run_ts
FROM pipeline_run_log r
JOIN dq_rule_result d
  ON r.pipeline_run_id = d.pipeline_run_id
WHERE r.start_time >= date_sub(current_timestamp(), 7)
  AND r.status <> 'SKIPPED'
  AND d.status = 'FAIL'
  AND d.severity IN ('CRITICAL', 'MAJOR')
ORDER BY r.start_time DESC, r.pipeline_run_id, d.severity, d.rule_name;


StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### 4) Gold DQ Monitoring
(Summary of failures from gold_dq_report)

In [18]:
%%sql
CREATE OR REPLACE VIEW v_monitor_gold_dq_fail_summary_7d AS
SELECT
  layer,
  table_name,
  rule_type,
  rule_name,
  column_name,
  COUNT(*) AS fail_events,
  MAX(to_timestamp(run_ts)) AS last_fail_ts
FROM dq_rule_result
WHERE to_timestamp(run_ts) >= date_sub(current_timestamp(), 7)
  AND status = 'FAIL'
GROUP BY layer, table_name, rule_type, rule_name, column_name
ORDER BY fail_events DESC, last_fail_ts DESC

StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 20, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### 5) Smoke Tests (Sample Queries)

In [19]:
%%sql
SELECT * FROM v_monitor_success_rate_7d LIMIT 50;
-- SELECT * FROM v_monitor_blocked_degraded_7d LIMIT 50;
-- SELECT * FROM v_monitor_top_failed_rules_7d LIMIT 20;
-- SELECT * FROM v_monitor_run_dq_fail_drilldown_7d LIMIT 50;


StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 21, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 11 fields>

In [20]:
%%sql
SELECT
pipeline_name,
table_name,
status,
dq_status,
start_time
FROM pipeline_run_log
WHERE pipeline_name = '04_5_gold_business_metrics'
ORDER BY start_time DESC
LIMIT 3;


StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 22, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 5 fields>

### (Optional) Test / Demo Data

In [21]:
%%sql
INSERT INTO pipeline_run_log (
  pipeline_run_id, 
  pipeline_name, 
  layer, 
  table_name, 
  status, 
  start_time, 
  end_time, 
  input_rows, 
  output_rows, 
  dq_status
)
VALUES (
  'test_blocked_run', 
  '03_2_dq_silver_run_all', 
  'silver', 
  'review_silver', 
  'BLOCKED', 
  current_timestamp(), 
  current_timestamp(), 
  NULL, 
  NULL, 
  'FAILED'
);


StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 23, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [22]:
%%sql
SELECT (*)
FROM pipeline_run_log
limit 3

StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 24, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 10 fields>

## 2. Runtime Intelligence Monitoring (Performance Layer)

### 1) Runtime SLA (Simple Rule)

In [23]:
%%sql
CREATE OR REPLACE VIEW v_monitor_sla_7d AS
SELECT
    r.pipeline_name,
    r.layer,
    date(r.start_time) AS run_date,
    UNIX_TIMESTAMP(r.end_time) - UNIX_TIMESTAMP(r.start_time) AS duration_sec,
    s.max_duration_sec,
    CASE
        WHEN UNIX_TIMESTAMP(r.end_time) - UNIX_TIMESTAMP(r.start_time) > s.max_duration_sec
        THEN 'SLA_BREACH'
        ELSE 'WITHIN_SLA'
    END AS sla_status
FROM pipeline_run_log r
JOIN pipeline_sla_config s
    ON r.pipeline_name = s.pipeline_name
WHERE r.start_time >= date_sub(current_timestamp(), 7)
    AND r.status NOT IN ('SKIPPED')
    AND r.end_time IS NOT NULL
    AND s.is_active = true

StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 25, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### 2) Runtime Percentiles (P50 / P95)

In [24]:
%%sql
CREATE OR REPLACE VIEW v_monitor_runtime_percentile_7d AS
WITH base AS (
    SELECT
        pipeline_name,
        layer,
        UNIX_TIMESTAMP(end_time) - UNIX_TIMESTAMP(start_time) AS duration_sec
    FROM pipeline_run_log
    WHERE start_time >= date_sub(current_timestamp(), 7)
        AND status NOT IN ('SKIPPED')
        AND end_time IS NOT NULL
)
SELECT
    pipeline_name,
    layer,
    percentile_approx(duration_sec, 0.5)  AS p50_duration,
    percentile_approx(duration_sec, 0.95) AS p95_duration,
    ROUND(AVG(duration_sec), 1)           AS avg_duration,
    COUNT(*)                              AS run_count
FROM base
WHERE duration_sec > 0
GROUP BY pipeline_name, layer

StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 26, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### 3) Runtime Drift Detection (3d vs 7d)

In [25]:
%%sql
CREATE OR REPLACE VIEW v_monitor_runtime_drift AS
WITH last_3d AS (
    SELECT
        pipeline_name,
        layer,
        AVG(UNIX_TIMESTAMP(end_time) - UNIX_TIMESTAMP(start_time)) AS avg_3d,
        COUNT(*) AS cnt_3d
    FROM pipeline_run_log
    WHERE start_time >= date_sub(current_timestamp(), 3)
        AND status NOT IN ('SKIPPED')
        AND end_time IS NOT NULL
    GROUP BY pipeline_name, layer
),
last_7d AS (
    SELECT
        pipeline_name,
        layer,
        AVG(UNIX_TIMESTAMP(end_time) - UNIX_TIMESTAMP(start_time)) AS avg_7d,
        COUNT(*) AS cnt_7d
    FROM pipeline_run_log
    WHERE start_time >= date_sub(current_timestamp(), 7)
        AND status NOT IN ('SKIPPED')
        AND end_time IS NOT NULL
    GROUP BY pipeline_name, layer
)
SELECT
    l3.pipeline_name,
    l3.layer,
    ROUND(l3.avg_3d, 1)                    AS avg_3d,
    ROUND(l7.avg_7d, 1)                    AS avg_7d,
    ROUND(l3.avg_3d - l7.avg_7d, 2)        AS duration_drift_sec
FROM last_3d l3
JOIN last_7d l7
    ON l3.pipeline_name = l7.pipeline_name
    AND l3.layer = l7.layer
WHERE l3.cnt_3d >= 1 AND l7.cnt_7d >= 2

StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 27, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [26]:
%%sql
SELECT (*) FROM v_monitor_runtime_drift


StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 28, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 5 fields>

## 3. SLA Monitoring (Config-Driven Governance Layer)
This section externalizes SLA thresholds into a configuration table (pipeline_sla_config),
allowing updates without modifying monitoring logic.

### 1) Create SLA Configuration Table

Table was created in 05_0_monitoring_init

### 2) Insert Active SLA Rules
Active rules were inserted in 05_0_monitoring_init

### 3) SLA Monitoring View (7-Day)

In [27]:
%%sql

SELECT (*)
FROM v_monitor_sla_7d
LIMIT 10;

StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 29, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 6 fields>

### 4) SLA Breach Summary (7-Day)

In [28]:
%%sql
CREATE OR REPLACE VIEW v_monitor_sla_summary_7d AS
SELECT
pipeline_name,
COUNT(*) AS total_runs,
SUM(
    CASE WHEN sla_status = 'SLA_BREACH' THEN 1 ELSE 0 END
) AS breach_cnt,
ROUND(
    100 * SUM(CASE WHEN sla_status = 'SLA_BREACH' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0),
    2
) AS breach_rate_pct
FROM v_monitor_sla_7d
GROUP BY pipeline_name;

StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 30, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### 5) Validate Outputs

In [29]:
%%sql
SELECT (*)
FROM v_monitor_sla_summary_7d
LIMIT 5

StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 31, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

## 4. Composite Run Health Score (KPI Layer)
### 1) Health Score View

In [30]:
%%sql
CREATE OR REPLACE VIEW v_monitor_run_health_score_7d AS
WITH success AS (
    -- Execution reliability: average success rate excluding SKIPPED runs
    SELECT
        pipeline_name,
        AVG(success_rate_excl_skipped_pct) AS success_rate,
        SUM(total_runs)                     AS total_runs
    FROM v_monitor_success_rate_7d
    GROUP BY pipeline_name
),
sla AS (
    -- SLA compliance score: 100 minus breach rate
    SELECT
        pipeline_name,
        100 - breach_rate_pct AS sla_score
    FROM v_monitor_sla_summary_7d
),
blocked AS (
    -- Blocked frequency: total blocked runs per pipeline
    SELECT
        pipeline_name,
        SUM(blocked_cnt) AS blocked_total
    FROM v_monitor_blocked_degraded_7d
    GROUP BY pipeline_name
),
critical_dq AS (
    -- Critical DQ failures: count of CRITICAL rule failures in drilldown
    SELECT
        pipeline_name,
        COUNT(*) AS critical_fail_cnt
    FROM v_monitor_run_dq_fail_drilldown_7d
    WHERE severity = 'CRITICAL'
    GROUP BY pipeline_name
)
SELECT
    s.pipeline_name,

    -- Component scores (each 0-100)
    ROUND(COALESCE(s.success_rate, 0), 2) AS reliability_score,
    ROUND(COALESCE(sla.sla_score, 100), 2) AS sla_score,
    ROUND(100 * (1.0 - COALESCE(b.blocked_total, 0) / NULLIF(s.total_runs, 0)), 2) AS blocked_score,
    ROUND(CASE WHEN COALESCE(c.critical_fail_cnt, 0) = 0 THEN 100 ELSE 0 END, 2) AS dq_score,

    -- Composite health score (weighted average, 0-100)
    -- Weights: reliability 40%, SLA 30%, blocked 20%, critical DQ 10%
    ROUND(
        0.4 * COALESCE(s.success_rate, 0)
        + 0.3 * COALESCE(sla.sla_score, 100)
        + 0.2 * 100 * (1.0 - COALESCE(b.blocked_total, 0) / NULLIF(s.total_runs, 0))
        + 0.1 * CASE WHEN COALESCE(c.critical_fail_cnt, 0) = 0 THEN 100 ELSE 0 END
    , 2) AS health_score,

    -- Health classification based on composite score
    CASE
        WHEN
            0.4 * COALESCE(s.success_rate, 0)
            + 0.3 * COALESCE(sla.sla_score, 100)
            + 0.2 * 100 * (1.0 - COALESCE(b.blocked_total, 0) / NULLIF(s.total_runs, 0))
            + 0.1 * CASE WHEN COALESCE(c.critical_fail_cnt, 0) = 0 THEN 100 ELSE 0 END
            >= 80 THEN 'HEALTHY'
        WHEN
            0.4 * COALESCE(s.success_rate, 0)
            + 0.3 * COALESCE(sla.sla_score, 100)
            + 0.2 * 100 * (1.0 - COALESCE(b.blocked_total, 0) / NULLIF(s.total_runs, 0))
            + 0.1 * CASE WHEN COALESCE(c.critical_fail_cnt, 0) = 0 THEN 100 ELSE 0 END
            >= 60 THEN 'WARNING'
        ELSE 'CRITICAL'
    END AS health_label

FROM success s
LEFT JOIN sla
    ON s.pipeline_name = sla.pipeline_name
LEFT JOIN blocked b
    ON s.pipeline_name = b.pipeline_name
LEFT JOIN critical_dq c
    ON s.pipeline_name = c.pipeline_name

StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 32, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### 2) Validation Query

In [31]:
%%sql
SELECT (*)
FROM v_monitor_run_health_score_7d
ORDER BY health_score DESC;

StatementMeta(, 8166b71e-7ed5-4f17-9d04-c9cf42ad1572, 33, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 7 fields>